In [1]:
# Cell 1 — Imports & folder setup
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
print('All libraries working!')
print('Folders ready: data/raw  data/processed')

All libraries working!
Folders ready: data/raw  data/processed


In [2]:
# Cell 2 — Download USGS global data (2020-2024, mag >= 4.5, limit 5000)
url_global = (
    'https://earthquake.usgs.gov/fdsnws/event/1/query?'
    'format=csv'
    '&starttime=2020-01-01'
    '&endtime=2024-01-01'
    '&minmagnitude=4.5'
    '&limit=5000'
    '&orderby=time'
)

print('Downloading global USGS data (2020-2024, mag>=4.5)...')
df_global = pd.read_csv(url_global)
df_global.to_csv('data/raw/usgs_global.csv', index=False)
print(f'Global dataset: {len(df_global)} events, {df_global.shape[1]} columns')
print(df_global[['time','place','mag','depth']].head(5))

Global dataset: 5000 events, 22 columns
                       time                              place  mag    depth
0  2023-12-31T19:19:14.340Z  180 km SW of Merizo Village, Guam  4.6   30.121
1  2023-12-31T17:04:48.597Z          south of the Fiji Islands  4.6  516.492
2  2023-12-31T17:00:47.300Z       7 km SW of Mati, Philippines  4.6   10.000
3  2023-12-31T16:36:23.043Z         27 km E of Nagarkot, Nepal  4.6   22.322
4  2023-12-31T15:23:14.621Z      258 km N of Tobelo, Indonesia  4.9   10.000


In [3]:
# Cell 3 — Download Morocco-region data (2000-2024, mag >= 4.0, bounding box)
url_morocco = (
    'https://earthquake.usgs.gov/fdsnws/event/1/query?'
    'format=csv'
    '&starttime=2000-01-01'
    '&endtime=2024-01-01'
    '&minmagnitude=4.0'
    '&minlatitude=27&maxlatitude=36'
    '&minlongitude=-13&maxlongitude=0'
    '&limit=1000'
    '&orderby=time'
)

print('Downloading Morocco-region data (2000-2024, mag>=4.0)...')
df_morocco = pd.read_csv(url_morocco)
df_morocco.to_csv('data/raw/morocco_earthquakes.csv', index=False)
print(f'Morocco dataset: {len(df_morocco)} events')
print(df_morocco[['time','place','mag','depth']].head(10))

Morocco dataset: 332 events
                       time                              place  mag   depth
0  2023-10-30T04:20:22.060Z  56 km WSW of Oukaïmedene, Morocco  4.2  10.000
1  2023-09-30T08:04:21.474Z  59 km WSW of Oukaïmedene, Morocco  4.0  10.000
2  2023-09-14T05:53:16.342Z  52 km WSW of Oukaïmedene, Morocco  4.6   8.722
3  2023-09-13T15:45:04.786Z     62 km NE of Taroudant, Morocco  4.2  10.000
4  2023-09-12T00:25:38.290Z  60 km WSW of Oukaïmedene, Morocco  4.0  10.000
5  2023-09-10T08:00:01.027Z  57 km WSW of Oukaïmedene, Morocco  4.2   9.003
6  2023-09-08T22:30:44.243Z  50 km WSW of Oukaïmedene, Morocco  4.9  10.000
7  2023-09-08T22:11:01.405Z                  Al Haouz, Morocco  6.8  19.000
8  2023-04-04T21:06:08.713Z  13 km SW of Bni Bouayach, Morocco  4.5  15.028
9  2023-02-12T08:36:45.805Z    9 km S of Bni Bouayach, Morocco  4.4  10.000


In [4]:
# Cell 4 — Merge, keep useful columns, assign risk labels
df_combined = pd.concat([df_global, df_morocco]).drop_duplicates(
    subset=['time','mag','depth']
).reset_index(drop=True)

df_clean = df_combined[['time','latitude','longitude','depth','mag','place']].copy()

def assign_risk(row):
    m = row['mag']
    if   m < 4.5: return 0   # Safe
    elif m < 5.5: return 1   # Watch
    elif m < 6.5: return 2   # Danger
    else:         return 3   # Evacuate

df_clean['risk_label'] = df_clean.apply(assign_risk, axis=1)
print('Risk label distribution:')
print(df_clean['risk_label'].value_counts().sort_index())
print('Label key: 0=Safe  1=Watch  2=Danger  3=Evacuate')

Risk label distribution:
risk_label
0     228
1    4741
2     329
3      31
Name: count, dtype: int64
Label key: 0=Safe  1=Watch  2=Danger  3=Evacuate


In [5]:
# Cell 5 — Generate 4000 synthetic Safe events (mag 2.0-4.4) to balance classes
np.random.seed(42)
n = 4000
safe_events = pd.DataFrame({
    'time':        'synthetic',
    'latitude':    np.random.uniform(27, 36, n),
    'longitude':   np.random.uniform(-13, 0, n),
    'depth':       np.random.uniform(5, 30, n),
    'mag':         np.random.uniform(2.0, 4.4, n),
    'place':       'Morocco region (synthetic)',
    'risk_label':  0
})

df_final = pd.concat([df_clean, safe_events]).reset_index(drop=True)
print(f'Final dataset size: {len(df_final)} rows')
print('Risk label distribution after balancing:')
print(df_final['risk_label'].value_counts().sort_index())

Final dataset size: 9329 rows
Risk label distribution after balancing:
risk_label
0    4228
1    4741
2     329
3      31
Name: count, dtype: int64


In [6]:
# Cell 6 — Add synthetic sensor readings correlated with magnitude
df_final['mag']   = pd.to_numeric(df_final['mag'],   errors='coerce')
df_final['depth'] = pd.to_numeric(df_final['depth'], errors='coerce')
df_final.dropna(subset=['mag','depth'], inplace=True)
df_final.reset_index(drop=True, inplace=True)
n = len(df_final)

df_final['vibration_hz'] = (
    df_final['mag'] * 1.8 + np.random.normal(0, 0.4, n)
).clip(0)

df_final['acceleration_m2s'] = (
    (10 - df_final['depth'].clip(1, 10)) * df_final['mag'] * 0.05
    + np.random.normal(0, 0.1, n)
).clip(0)

df_final['floor_level'] = np.random.randint(1, 11, n)

df_final['fire_sensor'] = np.where(
    df_final['risk_label'] >= 2,
    np.random.uniform(50, 300, n),
    np.random.uniform(0, 20, n)
)

print('Sensor columns added successfully.')
print(df_final.columns.tolist())

Sensor columns added successfully.
['time', 'latitude', 'longitude', 'depth', 'mag', 'place', 'risk_label', 'vibration_hz', 'acceleration_m2s', 'floor_level', 'fire_sensor']


In [7]:
# Cell 7 — Normalize features, drop NaN, save
features = ['mag','depth','vibration_hz','acceleration_m2s','floor_level','fire_sensor']

scaler = MinMaxScaler()
df_final[features] = scaler.fit_transform(df_final[features])
df_final.dropna(subset=features, inplace=True)

df_final.to_csv('data/processed/seismic_clean.csv', index=False)
print('=== PHASE 1 COMPLETE ===')
print(f'Total records  : {len(df_final)}')
print(f'Features       : {df_final.shape[1]} columns')
print(f'Missing values : {df_final.isnull().sum().sum()}')
print('\nRisk label counts:')
print(df_final['risk_label'].value_counts().sort_index())
print('\nFeature ranges (min / max):')
for f in features:
    print(f'  {f:<22}: {df_final[f].min():.3f} / {df_final[f].max():.3f}')

=== PHASE 1 COMPLETE ===
Total records  : 9329
Features       : 11 columns
Missing values : 0

Risk label counts:
risk_label
0    4228
1    4741
2     329
3      31
Name: count, dtype: int64

Feature ranges (min / max):
  mag                   : 0.000 / 1.000
  depth                 : 0.000 / 1.000
  vibration_hz          : 0.000 / 1.000
  acceleration_m2s      : 0.000 / 1.000
  floor_level           : 0.000 / 1.000
  fire_sensor           : 0.000 / 1.000
